# 1. Configurando as dependências e o MLFlow

In [ ]:
import sys
import warnings
from pathlib import Path

import pandas as pd
import mlflow

# Configurações para ignorar warnings
warnings.filterwarnings('ignore')

# Define a pasta raiz do repo e caso não ache a pasta src lá, adiciona a mesma
repo_root = Path("..").resolve()
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.append(str(src_path))

# Funções customizadas para configurar o MLflow Tracking e obter informações do ambiente
from tracking import (
    build_default_run_tags,
    configure_mlflow_tracking,
    get_owner,
    normalize_params,
 )

EXPERIMENT_NAME = "a ser preenchido"  # Placeholder, nao sera criado no MLflow
RANDOM_SEED = 42

MLFLOW_DB_PATH = repo_root / "data" / "mlflow_db" / "mlflow.db"
EXPERIMENT_TAGS = {
    "project": "ml-churn-prediction",
    "challenge_phase": "etapa_1_baselines",
    "business_domain": "telecom",
    "problem_type": "binary_classification",
    "target": "churn",
    "primary_metric": "f1",
    "dataset_name": "telco_customer_churn",
    "dataset_version": "v1_eda_encoded",
    "owner": get_owner(),
}

# Configura apenas o backend store. Nao cria experimento placeholder.
tracking_uri = f"sqlite:///{MLFLOW_DB_PATH.resolve().as_posix()}"
mlflow.set_tracking_uri(tracking_uri)

print("Tracking URI:", tracking_uri)
print("Experimento (placeholder):", EXPERIMENT_NAME)

## 1.1 Criando a função para treinar o modelo, calcular as métricas e logar tudo no MLFlow

In [1]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

def train_and_log_model(model, run_name, stage, model_family, 
                        sampling_strategy, X_tr, y_tr, X_te, y_te,
                        model_params=None, extra_tags=None):
    """Treina modelo, calcula métricas e loga no MLflow"""
    
    with mlflow.start_run(run_name=run_name):
        mlflow.set_tags(
            build_default_run_tags(
                stage=stage,
                model_family=model_family,
                sampling_strategy=sampling_strategy,
                extra_tags=extra_tags or {},
            )
        )
        
        if model_params:
            mlflow.log_params(normalize_params(model_params))
            
        model.fit(X_tr, y_tr)
        y_pred_train = model.predict(X_tr)
        y_pred_test = model.predict(X_te)
        
        metrics = {
            "train_accuracy": accuracy_score(y_tr, y_pred_train),
            "test_accuracy": accuracy_score(y_te, y_pred_test),
            "test_f1_score": f1_score(y_te, y_pred_test),
            "test_precision": precision_score(y_te, y_pred_test),
            "test_recall": recall_score(y_te, y_pred_test),
            "overfitting": accuracy_score(y_tr, y_pred_train) - accuracy_score(y_te, y_pred_test),
        }
        
        mlflow.log_metrics(metrics)
        print(f"=== {run_name.upper()} ===")
        for key, val in metrics.items():
            print(f"{key}: {val:.4f}")

# 2. Setando o baseline do projeto e configurando pela primeira vez o experimento do MLFlow

In [ ]:
EXPERIMENT_NAME = "project_baselines"
configure_mlflow_tracking(
    experiment_name=EXPERIMENT_NAME,
    db_path=MLFLOW_DB_PATH,
    experiment_tags=EXPERIMENT_TAGS,
)
print("Experimento ativo:", EXPERIMENT_NAME)

### 2.1 One-hot Encoding

A fim de facilitar a ingestão nos modelos de ML, utilizamos o __get_dummies__ do pandas para fazer o encoding do dataset

In [ ]:
df_clean = pd.read_csv("../data/processed/telco_customer_churn_eda_pre-processed.csv")
print("Shape do dataset:", df_clean.shape)


# One-hot encoding das variáveis categóricas relevantes
categorical_cols = ['Gender', 'Senior Citizen', 'Partner', 'Dependents', 'Phone Service', 'Multiple Lines', 
                    'Internet Service', 'Online Security', 'Online Backup', 'Device Protection', 
                    'Tech Support', 'Streaming TV', 'Streaming Movies', 'Contract', 'Paperless Billing', 
                    'Payment Method']
df_encoded = pd.get_dummies(df_clean, columns=categorical_cols, drop_first=True)

# Converter booleanos para inteiro
bool_cols = df_encoded.select_dtypes(include='bool').columns
df_encoded[bool_cols] = df_encoded[bool_cols].astype(int)

print(f"Shape após one-hot encoding: {df_encoded.shape}")
df_encoded.head()

# Salva uma cópia do dataset
df_encoded.to_csv("../data/processed/telco_customer_churn_eda_encoded.csv", index=False)

### 2.2 Separando o dataset entre treino e teste

In [ ]:
from sklearn.model_selection import train_test_split

# Dividindo o dataset em features (X) e target (y)
X = df_encoded.drop(columns=['target', 'CLTV', 'Total Charges'])
y = df_encoded['target']

# Divida em treino e teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = RANDOM_SEED)

### 2.3 Treinando o baseline nos dados do EDA

Regressão Logística

In [ ]:
log_reg_baseline_model = LogisticRegression(random_state = RANDOM_SEED, max_iter = 1000)

train_and_log_model(
    model = log_reg_baseline_model,
    X_tr = X_train,
    y_tr = y_train,
    X_te = X_test,
    y_te = y_test,
    run_name = "logistic_regression_baseline",
    stage = "baseline",
    model_family = "logistic_regression",
    sampling_strategy = "none",
    model_params={"model": "logistic_regression", "max_iter": 1000, "seed": RANDOM_SEED},
    extra_tags = {"run_purpose": "benchmark"}
)

DummyClassifier

In [ ]:
from sklearn.dummy import DummyClassifier

dummy_classifier_baseline_model = DummyClassifier(random_state=RANDOM_SEED)

train_and_log_model(
    model = dummy_classifier_baseline_model,
    X_tr = X_train,
    y_tr = y_train,
    X_te = X_test,
    y_te = y_test,
    run_name = "dummy_classifier_baseline",
    stage = "baseline",
    model_family = "dummy_classifier",
    sampling_strategy = "none",
    model_params={"model": "dummy_classifier", "strategy": "most_frequent", "seed": RANDOM_SEED},
    extra_tags = {"run_purpose": "benchmark"}
)

## 3. Testando parâmetros diferentes no modelo

In [ ]:
EXPERIMENT_NAME = "model_tuning"
configure_mlflow_tracking(
    experiment_name=EXPERIMENT_NAME,
    db_path=MLFLOW_DB_PATH,
    experiment_tags=EXPERIMENT_TAGS,
)
print("Experimento ativo:", EXPERIMENT_NAME)

In [ ]:
log_reg_balanced_model = LogisticRegression(random_state = RANDOM_SEED, max_iter = 1000, class_weight = "balanced")

train_and_log_model(
    model = log_reg_balanced_model,
    X_tr = X_train,
    y_tr = y_train,
    X_te = X_test,
    y_te = y_test,
    run_name = "logistic_regression_balanced",
    stage = "baseline",
    model_family = "logistic_regression",
    sampling_strategy = "class_weight_balanced",
    model_params={"model": "logistic_regression", 'class_weight': "balanced", "max_iter": 1000, "seed": RANDOM_SEED},
    extra_tags = {"run_purpose": "imbalance_handling"}
)

Devido ao desbalanceamento das classes (73% vs 27%), foi aplicada a estratégia de `class_weight='balanced'` na Regressão Logística, visando melhorar a capacidade do modelo em identificar clientes com churn. A abordagem resultou em aumento do recall, tornando o modelo mais sensível à classe minoritária.

| Métrica     | Baseline | Balanced | Mudança       |
| ----------- | -------- | -------- | ------------- |
| Accuracy    | 0.8098   | 0.7551   | ⬇️ (esperado) |
| Precision   | 0.7063   | 0.5459   | ⬇️            |
| Recall      | 0.5650   | 0.8175   | 🚀⬆️ MUITO    |
| F1 Score    | 0.6278   | 0.6547   | ⬆️            |
| Overfitting | 0.0072   | 0.0065   | ✅ ok          |

O modelo baseline apresentou bom desempenho geral, porém com recall limitado na identificação de churn.
Após aplicar class_weight='balanced', houve um aumento significativo no recall (de 56% para 81%), tornando o modelo mais eficaz na detecção de clientes propensos a churn, ainda que com redução na precisão — um trade-off aceitável para o contexto de negócio.

### 3.2 Utilizando SMOTE nos dados de treino

In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state = RANDOM_SEED)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

log_reg_balanced_smote_model = LogisticRegression(random_state = RANDOM_SEED, max_iter = 1000)

train_and_log_model(
    model = log_reg_balanced_smote_model,
    X_tr = X_train_smote,
    y_tr = y_train_smote,
    X_te = X_test,
    y_te = y_test,
    run_name = "logistic_regression_balanced_smote",
    stage = "baseline",
    model_family = "logistic_regression",
    sampling_strategy = "smote",
    model_params={"model": "logistic_regression", "max_iter": 1000, "seed": RANDOM_SEED},
    extra_tags = {"run_purpose": "imbalance_handling"}
)

| Modelo   | Precision | Recall   | F1       | Overfitting  |
| -------- | --------- | -------- | -------- | ------------ |
| Baseline | 0.70      | 0.56     | 0.62     | 0.007        |
| Balanced | 0.55      | **0.81** | **0.65** | 0.006        |
| SMOTE    | 0.58      | 0.69     | 0.63     | **0.069 ⚠️** |

Foram testadas diferentes estratégias para lidar com desbalanceamento, incluindo class_weight e SMOTE. A abordagem com class_weight='balanced' apresentou melhor desempenho, com maior recall e menor overfitting, sendo mais adequada para o problema de churn.